# Data dijet checks

Plot the reconstructed dijet $p_T^{ave}$ spectrum, the full $\eta_{CM}^{dijet}$ distribution in a configurable half-open $p_T^{ave}$ interval, and Forward/Backward ratios for each configured jet-acceptance cut. Full eta projections are normalized with ROOT to unit bin-content integral. Forward and Backward yields are not normalized before division, and `TH1::Divide` always uses standard independent-error propagation (never the binomial option).

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetPalette(ROOT.kBird)

from hist_analysis.python.histogram_io import load_histogram
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, draw_text_block, save_canvas, set_1d_style,
    set_legend_style, set_pad_style, style_single_panel_axes,
)

## Configuration

`PTAVE_BINS` configures an independent list of half-open eta-projection intervals for each trigger sample. `ETA_CUT_INDICES` controls the cuts shown together on both eta and F/B overlays. Set `FB_Y_RANGE` to a `(low, high)` tuple or to `None` for automatic scaling. The default `NORMALIZATION='integral'` implements `Scale(1.0 / Integral())`; `bin_width` and `none` are optional alternatives.

In [ ]:
DATA_DIR = Path(os.environ.get(
    'PPB_DATA_DIR', '/Users/gnigmat/cernbox/ana/pPb8160/exp',
))
OUTPUT_DIR = Path(os.environ.get(
    'DATA_CHECK_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'data_check',
))
PTAVE_DISPLAY_RANGE = (40.0, 500.0)
PTAVE_BINS = {
    'MinimumBias': [(60, 80), (80, 100), (100, 120), (120, 180)],
    'Jet60': [(80, 100), (100, 120), (120, 180), (180, 250)],
    'Jet80': [(100, 120), (120, 180), (180, 250), (300, 500)],
    'Jet100': [(120, 180), (180, 250), (300, 500)],
}
FB_Y_RANGE = (0.8, 1.15)  # for example (0.5, 1.5), or None for automatic scaling
PLOT_TEXT_SIZE = 0.028  # shared top-left annotation and legend size
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDICES = tuple(range(len(ETA_CUTS)))
REBIN_PTAVE = 1
REBIN_ETA = 2
NORMALIZATION = 'integral'  # integral (default), bin_width, or none
SAVE_PDF = True
SAVE_PNG = False
DRAW_GRID = True

DATA_FILES = {
    'Jet100': DATA_DIR / 'jet100_ak4_jetId.root',
    'Jet60': DATA_DIR / 'jet60_ak4_jetId.root',
    'Jet80': DATA_DIR / 'jet80_ak4_jetId.root',
    'MinimumBias': DATA_DIR / 'mb_ak4_jetId.root',
}

if NORMALIZATION not in {'integral', 'bin_width', 'none'}:
    raise ValueError('NORMALIZATION must be integral, bin_width, or none')
if set(PTAVE_BINS) != set(DATA_FILES):
    raise ValueError('PTAVE_BINS must define exactly the same samples as DATA_FILES')
for sample, intervals in PTAVE_BINS.items():
    if not intervals:
        raise ValueError(f'PTAVE_BINS[{sample!r}] must not be empty')
    if any(low >= high for low, high in intervals):
        raise ValueError(f'Invalid pTave interval for {sample}: {intervals}')
if FB_Y_RANGE is not None and FB_Y_RANGE[0] >= FB_Y_RANGE[1]:
    raise ValueError('FB_Y_RANGE must satisfy low < high')
if PLOT_TEXT_SIZE <= 0.0:
    raise ValueError('PLOT_TEXT_SIZE must be positive')
if not ETA_CUT_INDICES:
    raise ValueError('ETA_CUT_INDICES must not be empty')
if any(index < 0 or index >= len(ETA_CUTS) for index in ETA_CUT_INDICES):
    raise IndexError(f'Invalid ETA_CUT_INDICES: {ETA_CUT_INDICES}')

## ROOT histogram and drawing helpers

All projections and arithmetic remain ROOT histogram operations. The F/B construction hard-codes an empty divide option so binomial propagation cannot be enabled accidentally.

In [ ]:
def _safe_tag(text):
    return ''.join(character.lower() if character.isalnum() else '_'
                   for character in text).strip('_')

def _project(filename, key, observable, selection_range, name, rebin):
    source = load_histogram(str(filename), key)
    if not source.InheritsFrom('TH2'):
        raise TypeError(f'{key!r} in {filename} is not a TH2')
    source.Rebin2D(rebin if observable == 'ptave' else 1,
                   rebin if observable == 'eta' else 1)
    return project_semantic_th2(
        source, observable, selection_range, name=name,
    )

def _normalize_full_eta(histogram):
    normalized = histogram.Clone(f'{histogram.GetName()}_normalized')
    normalized.SetDirectory(0)
    if NORMALIZATION == 'none':
        return normalized
    integral = normalized.Integral()
    if integral <= 0.0:
        raise ValueError(f'Cannot normalize empty histogram {histogram.GetName()}')
    normalized.Scale(1.0 / integral)
    if NORMALIZATION == 'bin_width':
        normalized.Scale(1.0, 'width')
    return normalized

def _forward_backward(forward, backward, name):
    ratio = forward.Clone(name)
    ratio.SetDirectory(0)
    ratio.Divide(forward, backward, 1.0, 1.0, '')  # independent errors only
    return ratio

def _style_curve(histogram, style_index):
    set_1d_style(histogram, style_index)
    style_single_panel_axes(histogram)

def _draw_overlay(histograms, *, canvas_name, x_title, y_title,
                  annotations, x_range=None, y_range=None, log_y=False,
                  fixed_style=None, legend_position='top_right'):
    canvas = ROOT.TCanvas(
        canvas_name, '', DEFAULT_PLOT_STYLE.canvas_width,
        DEFAULT_PLOT_STYLE.canvas_height,
    )
    set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    canvas.SetLeftMargin(DEFAULT_PLOT_STYLE.single_panel_left_margin)
    canvas.SetBottomMargin(DEFAULT_PLOT_STYLE.single_panel_bottom_margin)
    canvas.SetLogy(log_y)
    if legend_position == 'bottom_left':
        legend_rows = (len(histograms) + 1) // 2
        legend_height = min(0.035 * legend_rows, 0.18)
        legend = ROOT.TLegend(0.22, 0.18, 0.57, 0.18 + legend_height)
        legend.SetNColumns(2)
    elif legend_position == 'top_right':
        legend_height = min(0.045 * len(histograms), 0.36)
        legend = ROOT.TLegend(0.62, 0.88 - legend_height, 0.88, 0.88)
    else:
        raise ValueError(f'Unsupported legend_position={legend_position!r}')
    set_legend_style(legend)
    legend.SetFillStyle(1001)
    legend.SetTextSize(PLOT_TEXT_SIZE)

    maximum = max(histogram.GetMaximum() for histogram in histograms.values())
    retained = []
    for curve_index, (label, histogram) in enumerate(histograms.items()):
        histogram.SetTitle('')
        histogram.GetXaxis().SetTitle(x_title)
        histogram.GetYaxis().SetTitle(y_title)
        if x_range is not None:
            histogram.GetXaxis().SetRangeUser(*x_range)
        if y_range is not None:
            histogram.GetYaxis().SetRangeUser(*y_range)
        if maximum > 0.0 and y_range is None:
            histogram.SetMaximum(maximum * (8.0 if log_y else 1.55))
        style_index = curve_index if fixed_style is None else fixed_style
        _style_curve(histogram, style_index)
        histogram.Draw('E1' if curve_index == 0 else 'E1 SAME')
        legend.AddEntry(histogram, label, 'p')
        retained.append(histogram)
    legend.Draw()
    text = draw_text_block(canvas, annotations, text_size=PLOT_TEXT_SIZE)
    canvas.Modified()
    canvas.Update()
    canvas._data_check_objects = [legend, *text, *retained]
    return canvas

def _analyze_data_interval(label, filename, ptave_range, *, include_ptave):
    filename = Path(filename)
    if not filename.exists():
        raise FileNotFoundError(f'Missing data ROOT file: {filename}')
    sample_tag = _safe_tag(label)
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    common_annotations = (
        'pPb 8.16 TeV data', label,
        f'{low:g} #leq p_{{T}}^{{ave}} < {high:g} GeV',
    )

    ptave = _project(
        filename, 'hRecoDijetPtEtaCM', 'ptave', None,
        f'h_{sample_tag}_ptave', REBIN_PTAVE,
    )
    eta_shapes = {}
    fb_ratios = {}
    raw_eta_integrals = {}
    for eta_cut_index in ETA_CUT_INDICES:
        eta_cut = ETA_CUTS[eta_cut_index]
        cut_label = f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}'
        eta = _project(
            filename, f'hRecoDijetPtEtaCM_{eta_cut_index}', 'eta',
            ptave_range, f'h_{sample_tag}_eta_{eta_cut_index}_{ptave_tag}',
            REBIN_ETA,
        )
        forward = _project(
            filename, f'hRecoDijetPtEtaForward_{eta_cut_index}', 'eta',
            ptave_range, f'h_{sample_tag}_forward_{eta_cut_index}_{ptave_tag}',
            REBIN_ETA,
        )
        backward = _project(
            filename, f'hRecoDijetPtEtaBackward_{eta_cut_index}', 'eta',
            ptave_range, f'h_{sample_tag}_backward_{eta_cut_index}_{ptave_tag}',
            REBIN_ETA,
        )
        raw_eta_integrals[eta_cut_index] = eta.Integral()
        eta_shapes[cut_label] = _normalize_full_eta(eta)
        fb_ratios[cut_label] = _forward_backward(
            forward, backward,
            f'h_{sample_tag}_fb_{eta_cut_index}_{ptave_tag}',
        )

    ptave_canvas = _draw_overlay(
        {label: ptave}, canvas_name=f'c_{sample_tag}_ptave',
        x_title='p_{T}^{ave} (GeV)', y_title='Dijets / bin',
        annotations=('pPb 8.16 TeV data', label),
        x_range=PTAVE_DISPLAY_RANGE, log_y=True, fixed_style=2,
    )
    eta_y_title = {
        'integral': '1/N dN/d#eta_{CM}^{dijet}',
        'bin_width': '1/N dN/d#eta_{CM}^{dijet}',
        'none': 'Dijets / bin',
    }[NORMALIZATION]
    eta_canvas = _draw_overlay(
        eta_shapes, canvas_name=f'c_{sample_tag}_eta_{ptave_tag}',
        x_title='#eta_{CM}^{dijet}', y_title=eta_y_title,
        annotations=common_annotations,
        x_range=(-max(ETA_CUTS[index] for index in ETA_CUT_INDICES) - 0.1,
                 max(ETA_CUTS[index] for index in ETA_CUT_INDICES) + 0.1),
    )
    fb_canvas = _draw_overlay(
        fb_ratios, canvas_name=f'c_{sample_tag}_fb_{ptave_tag}',
        x_title='|#eta_{CM}^{dijet}|', y_title='Forward / Backward',
        annotations=common_annotations,
        x_range=(0.0, max(ETA_CUTS[index] for index in ETA_CUT_INDICES) + 0.1),
        y_range=FB_Y_RANGE,
        legend_position='bottom_left',
    )

    if SAVE_PDF and include_ptave:
        save_canvas(ptave_canvas, OUTPUT_DIR / f'{sample_tag}_dijet_ptave.pdf',
                    save_png=SAVE_PNG)
    if SAVE_PDF:
        save_canvas(eta_canvas, OUTPUT_DIR / f'{sample_tag}_dijet_etaCM_ptave_{ptave_tag}.pdf',
                    save_png=SAVE_PNG)
        save_canvas(fb_canvas, OUTPUT_DIR / f'{sample_tag}_fb_etaCutOverlay_ptave_{ptave_tag}.pdf',
                    save_png=SAVE_PNG)

    print(label, 'raw eta integrals:', raw_eta_integrals)
    print(label, 'normalized eta integrals:',
          {cut: histogram.Integral() for cut, histogram in eta_shapes.items()})
    if include_ptave:
        display(ptave_canvas)
    display(eta_canvas)
    display(fb_canvas)
    return {
        'ptave': ptave, 'eta_shapes': eta_shapes,
        'forward_backward': fb_ratios, 'raw_eta_integrals': raw_eta_integrals,
        'ptave_canvas': ptave_canvas, 'eta_canvas': eta_canvas,
        'forward_backward_canvas': fb_canvas,
    }

def analyze_data_file(label, filename):
    results = {}
    if label not in PTAVE_BINS:
        raise KeyError(f'No PTAVE_BINS configuration for {label!r}')
    for interval_index, ptave_range in enumerate(PTAVE_BINS[label]):
        results[ptave_range] = _analyze_data_interval(
            label, filename, ptave_range, include_ptave=(interval_index == 0),
        )
    return results

## MinimumBias data

In [ ]:
mb_results = analyze_data_file('MinimumBias', DATA_FILES['MinimumBias'])

## Jet60 data

In [ ]:
jet60_results = analyze_data_file('Jet60', DATA_FILES['Jet60'])

## Jet80 data

In [ ]:
jet80_results = analyze_data_file('Jet80', DATA_FILES['Jet80'])

## Jet100 data

In [ ]:
jet100_results = analyze_data_file('Jet100', DATA_FILES['Jet100'])